In [1]:
# import pandas as pd
# import numpy as np
# import random
# from datetime import datetime, timedelta

# accounts = [f"A{i}" for i in range(200)]

# data = []

# start_date = datetime(2025,1,1)

# for _ in range(5000):

#     acc = random.choice(accounts)
#     cp = random.choice(accounts)

#     if acc == cp:
#         continue

#     timestamp = start_date + timedelta(
#         days=random.randint(0,70),
#         hours=random.randint(0,23)
#     )

#     amount = round(np.random.lognormal(mean=6, sigma=1),2)

#     data.append([
#         acc,
#         timestamp,
#         amount,
#         "OUT",
#         cp
#     ])

#     data.append([
#         cp,
#         timestamp,
#         amount,
#         "IN",
#         acc
#     ])

# df = pd.DataFrame(data, columns=[
# "Account","Timestamp","Amount","Type","Counterparty"
# ])

# df.to_csv("synthetic_transactions.csv", index=False)

In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# -----------------------
# CONFIG
# -----------------------

NUM_ACCOUNTS = 200
TARGET_TRANSACTIONS = 500
WEEKS = 12

start_date = datetime(2025,1,1)

accounts = [f"A{i}" for i in range(NUM_ACCOUNTS)]

# Account groups
normal_accounts = accounts[:120]
mule_accounts = accounts[120:150]
velocity_accounts = accounts[150:170]
dormant_accounts = accounts[170:190]
collector_accounts = accounts[190:200]

data = []

# -----------------------
# Helper function
# -----------------------

def random_timestamp():

    return start_date + timedelta(
        days=random.randint(0, WEEKS*7),
        hours=random.randint(0,23),
        minutes=random.randint(0,59)
    )

# -----------------------
# NORMAL ACCOUNTS
# -----------------------

for acc in normal_accounts:

    for _ in range(random.randint(20,40)):

        cp = random.choice(accounts)

        if cp == acc:
            continue

        amount = round(np.random.lognormal(6,1),2)

        ts = random_timestamp()

        data.append([acc,ts,amount,"OUT",cp])
        data.append([cp,ts,amount,"IN",acc])

# -----------------------
# VELOCITY ACCOUNTS
# -----------------------

for acc in velocity_accounts:

    for _ in range(random.randint(100,150)):

        cp = random.choice(accounts)

        if cp == acc:
            continue

        amount = random.randint(50,500)

        ts = random_timestamp()

        data.append([acc,ts,amount,"OUT",cp])
        data.append([cp,ts,amount,"IN",acc])

# -----------------------
# DORMANT ACCOUNTS
# -----------------------

for acc in dormant_accounts:

    for _ in range(random.randint(2,5)):

        cp = random.choice(accounts)

        if cp == acc:
            continue

        amount = random.randint(500,2000)

        ts = random_timestamp()

        data.append([acc,ts,amount,"OUT",cp])
        data.append([cp,ts,amount,"IN",acc])

# -----------------------
# MULE ACCOUNTS
# -----------------------

for acc in mule_accounts:

    for _ in range(random.randint(10,20)):

        sender = random.choice(accounts)

        if sender == acc:
            continue

        deposit = random.randint(20000,60000)

        ts = random_timestamp()

        # inflow spike
        data.append([acc,ts,deposit,"IN",sender])
        data.append([sender,ts,deposit,"OUT",acc])

        # fast outflow
        receiver = random.choice(accounts)

        withdraw = deposit * random.uniform(0.9,0.98)

        ts2 = ts + timedelta(minutes=random.randint(5,120))

        data.append([acc,ts2,withdraw,"OUT",receiver])
        data.append([receiver,ts2,withdraw,"IN",acc])

# -----------------------
# COLLECTOR (FAN-IN)
# -----------------------

for acc in collector_accounts:

    for _ in range(random.randint(50,80)):

        sender = random.choice(accounts)

        if sender == acc:
            continue

        amount = random.randint(1000,5000)

        ts = random_timestamp()

        data.append([sender,ts,amount,"OUT",acc])
        data.append([acc,ts,amount,"IN",sender])

# -----------------------
# BUILD DATAFRAME
# -----------------------

df = pd.DataFrame(data, columns=[
"Account",
"Timestamp",
"Amount",
"Type",
"Counterparty"
])

# Trim to target size if slightly larger
if len(df) > TARGET_TRANSACTIONS:
    df = df.sample(TARGET_TRANSACTIONS)

df = df.sort_values("Timestamp")

# -----------------------
# SAVE DATASET
# -----------------------

df.to_csv("synthetic_transactions.csv", index=False)

print("Dataset created")
print("Rows:",len(df))
print("Accounts:",df["Account"].nunique())

Dataset created
Rows: 500
Accounts: 182
